# 17 Figures

**Purpose.** Generate the result figures (ROC, confusion, margins, cohort, acquisition order).

**Thesis reference.** §5 (figures)

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:
# Writes PDFs into figures/ ready for \includegraphics.
import json, os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix,
                             precision_recall_curve, average_precision_score)
from sklearn.model_selection import RepeatedStratifiedKFold
from scipy.stats import mannwhitneyu

OUT = "figures"; os.makedirs(OUT, exist_ok=True)
PROFILES = "profiles"; MAXF = 3000

DEEP, TEAL, INK, MUTE, WARN = "#065A82", "#1C7293", "#16233A", "#6B7B8C", "#B85042"
rcParams.update({"font.family": "serif", "font.size": 9, "axes.linewidth": 0.8,
                 "axes.edgecolor": "#8899A6", "figure.dpi": 150,
                 "axes.spines.top": False, "axes.spines.right": False})

TASKS = {
 "detect_any_ckd":            ("labels/labels_detect_any_ckd.json",            "map_2d",     "Detection (any CKD)"),
 "severity_nondiabetic":      ("labels/labels_severity_nondiabetic.json",      "profile_1d", "Severity (non-diabetic)"),
 "positive_control_diabetes": ("labels/labels_positive_control_diabetes.json", "map_2d",     "Diabetes (positive control)"),
 "severity_all":              ("labels/labels_severity_all.json",              "map_2d",     "Severity (all patients)"),
}

ALL = json.load(open(f"{PROFILES}/profile_pids.json"))
_F = _C = None

def feats(kind, idx):
    global _F, _C
    if kind == "profile_1d":
        if _F is None: _F = np.load(f"{PROFILES}/mz_profiles.npy")
        M = _F[idx].astype(np.float64)
    else:
        if _C is None: _C = np.load(f"{PROFILES}/coarse_maps.npy")
        M = _C[idx].astype(np.float64).reshape(len(idx), -1)
    t = M.sum(1, keepdims=True); t[t == 0] = 1
    X = np.log1p(M / t * 1e6)
    return X[:, np.sort(np.argsort(X.var(0))[::-1][:MAXF])] if X.shape[1] > MAXF else X

mk = lambda: Pipeline([("s", StandardScaler()),
                       ("c", LogisticRegression(C=0.1, max_iter=3000,
                                                class_weight="balanced"))])

res = {}
for key, (lf, kind, nice) in TASKS.items():
    if not os.path.exists(lf):
        print(f"[skip] {lf}"); continue
    lab = json.load(open(lf))["labels"]
    idx = [i for i, p in enumerate(ALL) if p in lab]
    y = np.array([lab[ALL[i]] for i in idx])
    X = feats(kind, idx)
    oof = np.zeros(len(y)); cnt = np.zeros(len(y))
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=10,
                                          random_state=42).split(X, y):
        oof[te] += mk().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]; cnt[te] += 1
    oof /= cnt
    ids = np.array([int("".join(c for c in ALL[i] if c.isdigit())) for i in idx])
    u, _ = mannwhitneyu(ids[y == 1], ids[y == 0])
    a = u / ((y == 1).sum() * (y == 0).sum()); id_auc = max(a, 1 - a)
    res[key] = dict(y=y, p=oof, auc=roc_auc_score(y, oof), id_auc=id_auc,
                    nice=nice, n=len(y))
    print(f"{key:26s} n={len(y):4d}  AUC {res[key]['auc']:.3f}  ID {id_auc:.3f}")

# Figure 1: ROC curves, model vs identifier
fig, axes = plt.subplots(1, len(res), figsize=(3.0*len(res), 3.1))
if len(res) == 1: axes = [axes]
for ax, (k, r) in zip(axes, res.items()):
    fpr, tpr, _ = roc_curve(r["y"], r["p"])
    ax.plot(fpr, tpr, color=DEEP, lw=1.8, label=f"model  {r['auc']:.3f}")
    ax.plot([0, 1], [0, 1], color=MUTE, lw=0.8, ls=":")
    ax.axhline(-1)  # placeholder to keep legend order stable
    ax.plot([], [], color=WARN, lw=1.8, ls="--",
            label=f"patient ID  {r['id_auc']:.3f}")
    ax.set_title(f"{r['nice']}\n$n={r['n']}$", fontsize=9)
    ax.set_xlabel("false positive rate"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(loc="lower right", fontsize=7.5, frameon=False)
axes[0].set_ylabel("true positive rate")
fig.tight_layout(); fig.savefig(f"{OUT}/roc_curves.pdf", bbox_inches="tight"); plt.close(fig)
print("wrote roc_curves.pdf")

# Figure 2: confusion matrices at Youden's J
fig, axes = plt.subplots(1, len(res), figsize=(2.5*len(res), 2.7))
if len(res) == 1: axes = [axes]
for ax, (k, r) in zip(axes, res.items()):
    fpr, tpr, thr = roc_curve(r["y"], r["p"])
    t = thr[np.argmax(tpr - fpr)]
    cm = confusion_matrix(r["y"], (r["p"] >= t).astype(int))
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]}\n{cmn[i,j]:.0%}", ha="center", va="center",
                    fontsize=8, color="white" if cmn[i, j] > 0.55 else INK)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["neg", "pos"], fontsize=8)
    ax.set_yticklabels(["neg", "pos"], fontsize=8)
    ax.set_title(r["nice"], fontsize=8.5)
    ax.set_xlabel("predicted", fontsize=8)
    for s in ax.spines.values(): s.set_visible(False)
axes[0].set_ylabel("actual", fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/confusion.pdf", bbox_inches="tight"); plt.close(fig)
print("wrote confusion.pdf")

# Figure 3: margin over the identifier baseline
D = [("check_nondiabetic",112,0.077),("matched_nondiabetic",92,0.050),
     ("main_any_ckd",303,0.022),("detect_mckd",207,0.011),
     ("detect_nondiabetic",405,-0.007),("detect_any_ckd",596,-0.008),
     ("check_diabetic",191,-0.041),("severity_nondiabetic",359,0.122),
     ("severity_diabetic",151,0.095),("severity_all",510,0.206),
     ("dkd_vs_nondiabetic_mckd",121,0.130),("positive_control_diabetes",596,0.127)]
grp = {"severity_all":"severity","severity_nondiabetic":"severity","severity_diabetic":"severity",
       "positive_control_diabetes":"diabetes","dkd_vs_nondiabetic_mckd":"diabetes"}
D.sort(key=lambda r: r[2])
col = [DEEP if grp.get(t)=="severity" else (TEAL if grp.get(t)=="diabetes" else MUTE) for t,_,_ in D]
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.barh([t.replace("_"," ") for t,_,_ in D], [m for _,_,m in D], color=col, height=0.68)
ax.axvline(0, color=INK, lw=1.0)
ax.set_xlabel("margin over the patient-identifier baseline")
for i,(t,n,m) in enumerate(D):
    ax.text(m + (0.004 if m>=0 else -0.004), i, f"{m:+.3f}", va="center",
            ha="left" if m>=0 else "right", fontsize=7.5, color=INK)
import matplotlib.patches as mp
ax.legend(handles=[mp.Patch(color=MUTE,label="detection"),
                   mp.Patch(color=DEEP,label="severity"),
                   mp.Patch(color=TEAL,label="diabetes")],
          loc="lower right", fontsize=8, frameon=False)
ax.set_xlim(-0.09, 0.25)
fig.tight_layout(); fig.savefig(f"{OUT}/margins.pdf", bbox_inches="tight"); plt.close(fig)
print("wrote margins.pdf")

# Figure 4: cohort composition
fig, ax = plt.subplots(figsize=(5.4, 3.0))
cats = ["no CKD", "early CKD", "moderate CKD"]
dia  = np.array([40, 96, 55]); non = np.array([46, 293, 66])
ax.barh(cats, non, color=DEEP, label="no diabetes", height=0.6)
ax.barh(cats, dia, left=non, color=TEAL, label="diabetes", height=0.6)
for i,(a,b) in enumerate(zip(non,dia)):
    ax.text(a/2, i, str(a), ha="center", va="center", color="white", fontsize=8.5)
    ax.text(a+b/2, i, str(b), ha="center", va="center", color="white", fontsize=8.5)
    ax.text(a+b+8, i, f"{a+b}", va="center", fontsize=8.5, color=INK)
ax.set_xlabel("patients with an acquisition file")
ax.legend(fontsize=8, frameon=False, loc="lower right")
ax.set_xlim(0, 430)
fig.tight_layout(); fig.savefig(f"{OUT}/cohort.pdf", bbox_inches="tight"); plt.close(fig)
print("wrote cohort.pdf")

# Figure 5: acquisition order
fig, ax = plt.subplots(figsize=(6.4, 2.6))
lab = json.load(open("labels/labels_detect_any_ckd.json"))["labels"]
ids = np.array([int("".join(c for c in p if c.isdigit())) for p in ALL if p in lab])
ys  = np.array([lab[p] for p in ALL if p in lab])
ax.scatter(ids[ys==1], np.random.default_rng(0).normal(1,0.06,(ys==1).sum()),
           s=5, color=DEEP, alpha=0.45, label="CKD")
ax.scatter(ids[ys==0], np.random.default_rng(1).normal(0,0.06,(ys==0).sum()),
           s=7, color=WARN, alpha=0.75, label="control")
ax.set_yticks([0,1]); ax.set_yticklabels(["control","CKD"], fontsize=9)
ax.set_xlabel("patient identifier (enrollment order)")
ax.set_ylim(-0.4, 1.4); ax.legend(fontsize=8, frameon=False, loc="center left")
ax.spines["left"].set_visible(False)
fig.tight_layout(); fig.savefig(f"{OUT}/acquisition_order.pdf", bbox_inches="tight"); plt.close(fig)
print("wrote acquisition_order.pdf")

print(f"\nfive figures written to {OUT}/")

detect_any_ckd             n= 596  AUC 0.717  ID 0.719
severity_nondiabetic       n= 359  AUC 0.641  ID 0.519
positive_control_diabetes  n= 596  AUC 0.775  ID 0.643
severity_all               n= 510  AUC 0.741  ID 0.535
wrote roc_curves.pdf
wrote confusion.pdf
wrote margins.pdf
wrote cohort.pdf
wrote acquisition_order.pdf

five figures written to figures/
